# Phase 9 — 실습 문제 정답

정답 코드와 간단한 설명입니다.

## 1. 준비 (먼저 실행): 예제 이미지 + 헬퍼

In [ ]:
import numpy as np
rng=np.random.default_rng(3)
img=np.full((220,360),90,float); img[95:130,:]=185
img=np.clip(img+rng.normal(0,8,img.shape),0,255).astype(np.uint8)

def _profile(image,x,smooth=7):
    col=image[:,x].astype(float)
    pad=smooth//2; cp=np.pad(col,pad,mode='edge')
    return np.convolve(cp,np.ones(smooth)/smooth,mode='valid')

def _edges(prof):
    level=(np.percentile(prof,5)+np.percentile(prof,95))/2; xs=[]
    for i in range(len(prof)-1):
        y0,y1=prof[i],prof[i+1]
        if (y0-level)*(y1-level)<0: xs.append(i+(level-y0)/(y1-y0))
    return xs
print('준비 완료')

### 문제 1. 표준 measure() 완성
두께(px)에 scale 을 곱해 모으고, 통계를 dict 로 반환합니다.

In [ ]:
DEFAULT_PARAMS={'n_positions':20}
def measure(image, scale, params=None):
    p={**DEFAULT_PARAMS, **(params or {})}
    H,W=image.shape
    xs=np.linspace(int(W*0.1),int(W*0.9),p['n_positions']).astype(int)
    res=[]
    for x in xs:
        e=_edges(_profile(image,x))
        if len(e)<2:
            continue
        res.append((e[-1]-e[0])*scale)
    if not res:
        return {'status':'fail'}
    res=np.array(res)
    return {'mean_nm':float(res.mean()),'n':int(res.size),'status':'ok'}

r=measure(img,0.25)
assert r['status']=='ok' and abs(r['mean_nm']-8.75)<0.3
print('통과 —', r)

### 문제 2. 예외 처리
try/except 로 감싸 어떤 오류에도 fail 을 반환합니다.

In [ ]:
def safe_measure(image, scale, params=None):
    try:
        return measure(image, scale, params)
    except Exception as e:
        return {'status':'fail','error':str(e)}

assert safe_measure(None, 0.25)['status']=='fail'
assert safe_measure(img, 0.25)['status']=='ok'
print('통과')

### 문제 3. 재현성 확인
같은 입력 → 같은 출력. dict 비교로 확인합니다.

In [ ]:
same = (measure(img,0.25) == measure(img,0.25))

assert same is True
print('통과 — 재현성 확인')

---
정답 확인을 마칩니다.